<div class="alert alert-block" style="background-color: #51585eff; border-color: #42474aff; color: white;">
<center> <h1> Syntravel - Travel Diaries Generation </h1> </center> <br>
<center> <h2> Travel Diary Generation </h2></center>


# Table of Contents

**1. [Importing Libraries and Loading Data](#importing-libraries-and-data)**  
   - [1.1 Importing Libraries](#importing-libraries)  
   - [1.2 Loading and Reading Data](#loading-and-reading-data)  

**2. [Framework Ablations](#frame)**
- [2.1 Full Generation](#full) 
- [2.2 No Plan](#noplan) 
- [2.3 No persona & No patterns](#nopersona) 

**3. [Model Ablations](#model)**
- [3.1 OpenAi Full Generation](#openai) 
- [3.2 Qwen Full Generation](#qwen) 
- [3.3 Anthropic Full Generation](#claude) 

**3. [Mode and Distance Ablations](#modeabl)**
- [3.1 Mode Ablation](#mode) 
- [3.2 Distance Ablation](#distance) 
- [3.3 Mode & Distance Ablation](#modedistance) 




# 1. Importing Libraries and Loading Data  <a class="anchor" id="importing-libraries-and-data"></a>

## 1.1. Importing Libraries <a class="anchor" id="importing-libraries"></a>

In [ ]:
import os
if not os.path.isdir('llm_config'):
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
import re
import sys
import json

## functions imports
from llm_config.llm_config import *
from Helpers.trajectory_generation import *
from Helpers.trajectory_generation_ablations import *
from Helpers.cot_patterns import *


%config InlineBackend.figure_format = 'retina' 

## 1.2. Loading and Reading Data <a class="anchor" id="loading-and-reading-data"></a>

In [3]:
with open('Json_files/persona_objects.json', encoding='utf-8') as f:
      persona_objects = json.load(f)

print(f'Loaded {len(persona_objects)} persona objects')

Loaded 190 persona objects


In [4]:
with open('Json_files/cot_results.json', encoding='utf-8') as f:
      cot_results = json.load(f)

patterns = extract_final_patterns(cot_results)
print(f'Loaded {len(patterns)} patterns')

Loaded 190 patterns


In [8]:
# Originally, we were going to generaye weekday and weekend personas separately, but we end up only generating weekday personas
weekday_personas = [p for p in persona_objects if p['day_type'] == 'weekday']
missing = [p['group_key'] for p in weekday_personas if p['group_key'] not in patterns]

print(f'Weekday personas : {len(weekday_personas)}')
print(f'Missing patterns : {len(missing)}')
if missing:
      print('Missing keys:', missing)

Weekday personas : 113
Missing patterns : 0


# 2. Framework Ablations  <a class="anchor" id="frame"></a>

All generations are ran with 5 seeds: 3,7,13,27,42

In this Ablation we aim to undertstand the impact of the different components of the framework

## 2.1. Full Generation  <a class="anchor" id="full"></a>


In [22]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/variance/full_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'groq_generation',
        n_per_group     = 5,
        seed            = seed,
        use_persona     = True,
        use_daily_plan  = True,
        use_pattern     = True,
    )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] employed | 51-66 | weekday | Income=High income — generating 5
  syn_weekday_0531 
  syn_weekday_0532 
  syn_weekday_0533 
  syn_weekday_0534 
  syn_weekday_0535 
  -> saved (5 total for employed | 51-66 | weekday | Income=High income)
[5/113] employed | 51-66 | weekday | Income=Low income — generating 5
  syn_weekday_0536 [ZERO_TRIP]
  syn_weekday_0537 
  syn_weekday_0538 
  syn_weekday_0539 
  syn_weekday_0540 [ZERO_TRIP]
  -> saved (5 total for employed | 51-66 | weekday | Income=Low income)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] inactive | 51-66 | weekday | Income=Above median — generating 5
  syn_weekday_0541 [ZERO_TRIP]
  syn_weekday_0542 [ZERO_TRIP]
  syn_weekday_0543 
  syn_weekday_0544 
  syn_weekday_0545 
  -> saved (5 total for inactiv

## 2.2. No Plan  <a class="anchor" id="noplan"></a>


In [23]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/variance/no_plan_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'groq_generation',
        n_per_group     = 5,
        seed            = seed,
        use_persona     = True,
        use_daily_plan  = False,
        use_pattern     = True,
    )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] employed | 51-66 | weekday | Income=High income — generating 5
  syn_weekday_0531 
  syn_weekday_0532 
  syn_weekday_0533 
  syn_weekday_0534 [ZERO_TRIP]
  syn_weekday_0535 
  -> saved (5 total for employed | 51-66 | weekday | Income=High income)
[5/113] employed | 51-66 | weekday | Income=Low income — generating 5
  syn_weekday_0536 
  syn_weekday_0537 
  syn_weekday_0538 
  syn_weekday_0539 
  syn_weekday_0540 
  -> saved (5 total for employed | 51-66 | weekday | Income=Low income)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] inactive | 51-66 | weekday | Income=Above median — generating 5
  syn_weekday_0541 
  syn_weekday_0542 
  syn_weekday_0543 
  syn_weekday_0544 
  syn_weekday_0545 
  -> saved (5 total for inactive | 51-66 | weekday | Income=Abov

## 2.3. No persona & No patterns  <a class="anchor" id="nopersona"></a>

In [24]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/variance/no_patterns_personas_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'groq_generation',
        n_per_group     = 5,
        seed            = seed,
        use_persona     = False,
        use_daily_plan  = True,
        use_pattern     = False,
    )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] SKIP employed | 51-66 | weekday | Income=High income (already 5)
[5/113] SKIP employed | 51-66 | weekday | Income=Low income (already 5)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] SKIP inactive | 51-66 | weekday | Income=Above median (already 5)
[8/113] SKIP inactive | 51-66 | weekday | Income=Below median (already 5)
[9/113] SKIP inactive | 51-66 | weekday | Income=High income (already 5)
[10/113] SKIP inactive | 51-66 | weekday | Income=Low income (already 5)
[11/113] SKIP inactive | 51-66 | weekday | Income=Median (already 5)
[12/113] SKIP employed | 36-40 | weekday | Income=Above median (already 5)
[13/113] SKIP employed | 36-40 | weekday | Income=Below median (already 5)
[14/113] SKIP employed | 36-40 | weekday | Income=High income (already 5)
[

# 3. Model Ablations  <a class="anchor" id="model"></a>

All generations are ran with 5 seeds: 3,7,13,27,42

Except QWEN, due to very poor results seeds experiment were not done on this model. 

And Claude haiku who was also not part of the variance test due to the cost implications. Since the competition was between LLama and Open ai, the variance analysis focused on them 

## 3.1. OpenAi Full Generation  <a class="anchor" id="openai"></a>


In [25]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/variance/openai_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'openai',
        n_per_group     = 5,
        seed            = seed,
        use_persona     = True,
        use_daily_plan  = True,
        use_pattern     = True,
    )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] employed | 51-66 | weekday | Income=High income — generating 5
  syn_weekday_0669 
  syn_weekday_0670 
  syn_weekday_0671 
  syn_weekday_0672 
  syn_weekday_0673 
  -> saved (5 total for employed | 51-66 | weekday | Income=High income)
[5/113] employed | 51-66 | weekday | Income=Low income — generating 5
  syn_weekday_0674 
  syn_weekday_0675 
  syn_weekday_0676 
  syn_weekday_0677 
  syn_weekday_0678 
  -> saved (5 total for employed | 51-66 | weekday | Income=Low income)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] inactive | 51-66 | weekday | Income=Above median — generating 5
  syn_weekday_0679 
  syn_weekday_0680 [ZERO_TRIP]
  syn_weekday_0681 
  syn_weekday_0682 [ZERO_TRIP]
  syn_weekday_0683 
  -> saved (5 total for inactive | 51-66 | weekday | 

## 3.2. Qwen Full Generation  <a class="anchor" id="qwen"></a>


In [29]:
## Not used in the final version due to very bad results compared to the other models 
OUTPUT_FILE = 'Json_files/trajectories_weekday_full_qwen.json'

results = run_generation_pipeline(
      persona_objects = weekday_personas,
      patterns        = patterns,
      output_file     = OUTPUT_FILE,
      day_type        = 'weekday',
      provider        = 'qwen',
      n_per_group     = 5,
      seed            = 42,
      use_persona     = True,
      use_daily_plan  = True,
      use_pattern     = True,
  )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] SKIP employed | 51-66 | weekday | Income=High income (already 5)
[5/113] SKIP employed | 51-66 | weekday | Income=Low income (already 5)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] SKIP inactive | 51-66 | weekday | Income=Above median (already 5)
[8/113] SKIP inactive | 51-66 | weekday | Income=Below median (already 5)
[9/113] SKIP inactive | 51-66 | weekday | Income=High income (already 5)
[10/113] SKIP inactive | 51-66 | weekday | Income=Low income (already 5)
[11/113] SKIP inactive | 51-66 | weekday | Income=Median (already 5)
[12/113] SKIP employed | 36-40 | weekday | Income=Above median (already 5)
[13/113] SKIP employed | 36-40 | weekday | Income=Below median (already 5)
[14/113] SKIP employed | 36-40 | weekday | Income=High income (already 5)
[

## 3.3. Anthropic Full Generation  <a class="anchor" id="claude"></a>


In [30]:
OUTPUT_FILE = 'Json_files/trajectories_weekday_full_anthropic.json'

results = run_generation_pipeline(
      persona_objects = weekday_personas,
      patterns        = patterns,
      output_file     = OUTPUT_FILE,
      day_type        = 'weekday',
      provider        = 'claude',
      n_per_group     = 5,
      seed            = 42,
      use_persona     = True,
      use_daily_plan  = True,
      use_pattern     = True,
  )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] SKIP employed | 51-66 | weekday | Income=High income (already 5)
[5/113] SKIP employed | 51-66 | weekday | Income=Low income (already 5)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] SKIP inactive | 51-66 | weekday | Income=Above median (already 5)
[8/113] SKIP inactive | 51-66 | weekday | Income=Below median (already 5)
[9/113] SKIP inactive | 51-66 | weekday | Income=High income (already 5)
[10/113] SKIP inactive | 51-66 | weekday | Income=Low income (already 5)
[11/113] SKIP inactive | 51-66 | weekday | Income=Median (already 5)
[12/113] SKIP employed | 36-40 | weekday | Income=Above median (already 5)
[13/113] SKIP employed | 36-40 | weekday | Income=Below median (already 5)
[14/113] SKIP employed | 36-40 | weekday | Income=High income (already 5)
[

In [29]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/variance/claude_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'claude',
        n_per_group     = 5,
        seed            = seed,
        use_persona     = True,
        use_daily_plan  = True,
        use_pattern     = True,
    )

[1/113] homemaker | 65+ | weekday — generating 5
  syn_weekday_0001 [ZERO_TRIP]
  syn_weekday_0002 
  syn_weekday_0003 [ZERO_TRIP]
  syn_weekday_0004 
  syn_weekday_0005 
  -> saved (5 total for homemaker | 65+ | weekday)
[2/113] employed | 51-66 | weekday | Income=Above median — generating 5
  syn_weekday_0006 
  syn_weekday_0007 
  syn_weekday_0008 
  syn_weekday_0009 
  syn_weekday_0010 
  -> saved (5 total for employed | 51-66 | weekday | Income=Above median)
[3/113] employed | 51-66 | weekday | Income=Below median — generating 5
  syn_weekday_0011 
  syn_weekday_0012 
  syn_weekday_0013 
  syn_weekday_0014 
  syn_weekday_0015 
  -> saved (5 total for employed | 51-66 | weekday | Income=Below median)
[4/113] employed | 51-66 | weekday | Income=High income — generating 5
  syn_weekday_0016 
  syn_weekday_0017 [ZERO_TRIP]
  syn_weekday_0018 
  syn_weekday_0019 
  syn_weekday_0020 
  -> saved (5 total for employed | 51-66 | weekday | Income=High income)
[5/113] employed | 51-66 | week


# 4. Mode and Distance Ablations  <a class="anchor" id="newabl"></a>

All generations are ran with 5 seeds: 3,7,13,27,42

With this ablations we aim to understand how the results change if we let the LLM choose the value through its own reasoning, rather than pre-sampling it from the persona's empirical ODiN distribution. The tests are: mode alone, distance alone, and both together


## 4.1. Mode Ablation  <a class="anchor" id="mode"></a>



In [26]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline_ablations(
        persona_objects, patterns,
        output_file=f"Json_files/variance/mode_ablation_seed{seed}.json",
        day_type="weekday",
        provider="groq_generation",
        n_per_group=5,
        seed=seed,
        llm_mode=True,
        llm_distance=False,
    )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] employed | 51-66 | weekday | Income=High income — generating 5
  syn_weekday_0531 
  syn_weekday_0532 
  syn_weekday_0533 [ZERO_TRIP]
  syn_weekday_0534 [ZERO_TRIP]
  syn_weekday_0535 
  -> saved (5 total for employed | 51-66 | weekday | Income=High income)
[5/113] employed | 51-66 | weekday | Income=Low income — generating 5
  syn_weekday_0536 
  syn_weekday_0537 
  syn_weekday_0538 
  syn_weekday_0539 
  syn_weekday_0540 [ZERO_TRIP]
  -> saved (5 total for employed | 51-66 | weekday | Income=Low income)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] inactive | 51-66 | weekday | Income=Above median — generating 5
  syn_weekday_0541 
  syn_weekday_0542 
  syn_weekday_0543 
  syn_weekday_0544 
  syn_weekday_0545 [ZERO_TRIP]
  -> saved (5 total for inactiv

## 4.2. Distance Ablation  <a class="anchor" id="distance"></a>

In [27]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline_ablations(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/variance/distance_ablation_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'groq_generation',
        n_per_group     = 5,
        seed            = seed,
        llm_mode        = False,
        llm_distance    = True,
    )


[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] employed | 51-66 | weekday | Income=High income — generating 5
  syn_weekday_0531 
  syn_weekday_0532 
  syn_weekday_0533 [ZERO_TRIP]
  syn_weekday_0534 
  syn_weekday_0535 
  -> saved (5 total for employed | 51-66 | weekday | Income=High income)
[5/113] employed | 51-66 | weekday | Income=Low income — generating 5
  syn_weekday_0536 
  syn_weekday_0537 
  syn_weekday_0538 
  syn_weekday_0539 [ZERO_TRIP]
  syn_weekday_0540 
  -> saved (5 total for employed | 51-66 | weekday | Income=Low income)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] inactive | 51-66 | weekday | Income=Above median — generating 5
  syn_weekday_0541 [ZERO_TRIP]
  syn_weekday_0542 
  syn_weekday_0543 
  syn_weekday_0544 
  syn_weekday_0545 [ZERO_TRIP]
  -> saved (5 total for inactiv

## 4.3. Mode & Distance Ablation  <a class="anchor" id="modedistance"></a>

In [28]:
for seed in [3, 7, 13, 27, 42]:
    run_generation_pipeline_ablations(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/variance/mode_distance_ablation_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'groq_generation',
        n_per_group     = 5,
        seed            = seed,
        llm_mode        = True,
        llm_distance    = True,
    )

[1/113] SKIP homemaker | 65+ | weekday (already 5)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 5)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 5)
[4/113] employed | 51-66 | weekday | Income=High income — generating 5
  syn_weekday_0537 
  syn_weekday_0538 
  syn_weekday_0539 
  syn_weekday_0540 
  syn_weekday_0541 
  -> saved (5 total for employed | 51-66 | weekday | Income=High income)
[5/113] employed | 51-66 | weekday | Income=Low income — generating 5
  syn_weekday_0542 [ZERO_TRIP]
  syn_weekday_0543 [ZERO_TRIP]
  syn_weekday_0544 
  syn_weekday_0545 
  syn_weekday_0546 
  -> saved (5 total for employed | 51-66 | weekday | Income=Low income)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 5)
[7/113] inactive | 51-66 | weekday | Income=Above median — generating 5
  syn_weekday_0547 
  syn_weekday_0548 
  syn_weekday_0549 
  syn_weekday_0550 
  syn_weekday_0551 
  -> saved (5 total for inactive | 51-66 | weekday | 

# 5. Big run  <a class="anchor" id="bigrun"></a>

This big run is generated to validate the spatial allocation and produce significant visuals of it 

In [ ]:
ROTTERDAM_SEEDS = [3,7,42]                 # [3, 7, 13, 27, 42] for the full sweep
ROTTERDAM_N_PER_GROUP = 30

for seed in ROTTERDAM_SEEDS:
    run_generation_pipeline(
        persona_objects = weekday_personas,
        patterns        = patterns,
        output_file     = f"Json_files/trajectories_weekday_big_seed{seed}.json",
        day_type        = 'weekday',
        provider        = 'groq_generation',
        n_per_group     = ROTTERDAM_N_PER_GROUP,
        seed            = seed,
        use_persona     = True,
        use_daily_plan  = True,
        use_pattern     = True,
    )

[1/113] SKIP homemaker | 65+ | weekday (already 30)
[2/113] SKIP employed | 51-66 | weekday | Income=Above median (already 30)
[3/113] SKIP employed | 51-66 | weekday | Income=Below median (already 30)
[4/113] SKIP employed | 51-66 | weekday | Income=High income (already 30)
[5/113] SKIP employed | 51-66 | weekday | Income=Low income (already 30)
[6/113] SKIP employed | 51-66 | weekday | Income=Median (already 30)
[7/113] SKIP inactive | 51-66 | weekday | Income=Above median (already 30)
[8/113] SKIP inactive | 51-66 | weekday | Income=Below median (already 30)
[9/113] SKIP inactive | 51-66 | weekday | Income=High income (already 30)
[10/113] SKIP inactive | 51-66 | weekday | Income=Low income (already 30)
[11/113] SKIP inactive | 51-66 | weekday | Income=Median (already 30)
[12/113] SKIP employed | 36-40 | weekday | Income=Above median (already 30)
[13/113] SKIP employed | 36-40 | weekday | Income=Below median (already 30)
[14/113] SKIP employed | 36-40 | weekday | Income=High income 